# AI PILOT LLM — Fine-Tune on Google Colab

**Model:** Qwen3-8B (4-bit QLoRA via Unsloth)
**Dataset:** 14,390 pairs (12,951 train / 1,439 val)
**GPU:** T4 (free) or A100 (Colab Pro)
**Time:** ~3-4h on T4, ~1.5h on A100

## Quick Start
1. Runtime → Change runtime type → **T4 GPU** (or A100 for Pro)
2. Run all cells (Ctrl+F9)
3. Download model from Google Drive when done

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3 if hasattr(props, 'total_memory') else 0
print(f"VRAM: {vram_gb:.1f} GB")

In [ ]:
# Cell 2: Install Unsloth + dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
print("Dependencies installed!")

In [ ]:
# Cell 3: Clone repository + dataset
import os
os.chdir('/content')

if not os.path.exists('ai-pilot-llm'):
    !git clone https://github.com/abcprocessus/ai-pilot-llm.git
os.chdir('ai-pilot-llm')

# Verify dataset
!wc -l datasets/train.jsonl datasets/val.jsonl

In [ ]:
# Cell 4: Load model
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

# Add LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {trainable:,} trainable / {total:,} total ({trainable/total*100:.2f}%)")

In [ ]:
# Cell 5: Load dataset
import json
from datasets import Dataset

def read_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_rows = read_jsonl('datasets/train.jsonl')
val_rows = read_jsonl('datasets/val.jsonl')
print(f"Dataset: {len(train_rows)} train, {len(val_rows)} val")

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text}

train_ds = train_ds.map(format_chat)
val_ds = val_ds.map(format_chat)
print(f"Formatted. Sample:\n{train_ds[0]['text'][:300]}...")

In [ ]:
# Cell 6: Train!
from trl import SFTTrainer, SFTConfig

EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-4
OUTPUT_DIR = 'checkpoints/ai-pilot-llm-v1'

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(train_ds) // effective_batch

print(f"Training: {EPOCHS} epochs, {steps_per_epoch} steps/epoch, {steps_per_epoch * EPOCHS} total steps")
print(f"Effective batch size: {effective_batch}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=steps_per_epoch // 2,
    save_strategy='steps',
    save_steps=steps_per_epoch,
    save_total_limit=2,
    bf16=True,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

print("Starting training...")
result = trainer.train()
print(f"\nDone! Loss: {result.training_loss:.4f}, Runtime: {result.metrics.get('train_runtime', 0):.0f}s")

In [ ]:
# Cell 7: Save LoRA adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save metrics
metrics = {
    'train_loss': result.training_loss,
    'train_runtime_sec': result.metrics.get('train_runtime', 0),
    'train_samples': len(train_ds),
    'val_samples': len(val_ds),
    'base_model': BASE_MODEL,
    'epochs': EPOCHS,
}
with open(f'{OUTPUT_DIR}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"LoRA adapter saved: {OUTPUT_DIR}")
!cat {OUTPUT_DIR}/training_metrics.json

In [ ]:
# Cell 8: Merge LoRA into base model (for vLLM deployment)
MERGED_DIR = 'checkpoints/ai-pilot-llm-v1-merged'
print(f"Merging LoRA into base model -> {MERGED_DIR}")
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f"Merged model saved: {MERGED_DIR}")
!du -sh {MERGED_DIR}

In [ ]:
# Cell 9: Export GGUF (for Ollama)
GGUF_DIR = 'checkpoints/ai-pilot-llm-v1-gguf'
print(f"Exporting GGUF (Q4_K_M) -> {GGUF_DIR}")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method='q4_k_m')
print(f"GGUF saved: {GGUF_DIR}")
!ls -lh {GGUF_DIR}/

In [ ]:
# Cell 10: Quick test (inference without vLLM)
FastLanguageModel.for_inference(model)

test_prompts = [
    "Привет! Расскажи о себе.",
    "Как подключить CRM к AI агенту?",
    "Чем отличается тариф Lite от Business?",
    "Помоги составить контент-план для Instagram.",
    "Какие документы нужны для регистрации ИП в Беларуси?",
]

system = "Ты AI PILOT — платформа AI-сотрудников для бизнеса. Отвечай профессионально, конкретно, на русском языке."

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    outputs = model.generate(inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:300]}")
    print("-" * 60)

In [ ]:
# Cell 11: Upload to Google Drive (for download)
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DIR = '/content/drive/MyDrive/AI_PILOT_LLM'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy GGUF (smaller, ~5GB)
print("Copying GGUF to Google Drive...")
shutil.copytree(GGUF_DIR, f'{DRIVE_DIR}/ai-pilot-llm-v1-gguf', dirs_exist_ok=True)

# Copy LoRA adapter (small, ~100MB)
print("Copying LoRA adapter to Google Drive...")
shutil.copytree(OUTPUT_DIR, f'{DRIVE_DIR}/ai-pilot-llm-v1-lora', dirs_exist_ok=True)

# Copy metrics
shutil.copy(f'{OUTPUT_DIR}/training_metrics.json', f'{DRIVE_DIR}/training_metrics.json')

print(f"\nAll saved to Google Drive: {DRIVE_DIR}")
!ls -lh {DRIVE_DIR}/

## Next Steps

1. **Download** merged model or GGUF from Google Drive
2. **Deploy vLLM** on GPU server:
   ```bash
   docker run --gpus all -p 8000:8000 \
     -v /path/to/ai-pilot-llm-v1-merged:/model \
     vllm/vllm-openai \
     --model /model --served-model-name ai-pilot-llm-1.0 --port 8000
   ```
3. **Or use Ollama** with GGUF:
   ```bash
   ollama create ai-pilot-llm -f Modelfile
   ollama run ai-pilot-llm
   ```
4. **Set** `LOCAL_LLM_URL=http://gpu-server:8000` in Railway env
5. **Test** via FastAPI: `POST /api/v1/llm/chat` with `{"provider": "local"}`